# Data Cleaning and EDA Pipeline

In [1]:
# Import libraries and load the dataset
import pandas as pd
import numpy as np

df = pd.read_csv("Product_SupplyChain_Dataset.csv")

print("Shape of dataset:", df.shape)
df.head()

Shape of dataset: (4000, 11)


,Product_ID,Product_Category,Brand_Tier,Unit_Price,Discount_Percentage,Advertising_Spend,Competitor_Price,Store_Coverage_Count,Customer_Rating,Inventory_Availability_Percentage,Monthly_Units_Sold
0,PID000001,Beauty & Personal Care,Premium,58.46,1.7,9748.70,47.46,88.0,4.3,87.9,389
1,PID000002,Grocery,Mid-range,15.98,17.9,5279.12,48.54,215.0,3.8,99.5,450
2,PID000003,Beauty & Personal Care,Budget,16.35,17.7,5165.84,59.57,481.0,3.5,96.7,594
3,PID000004,Grocery,Mid-range,11.31,34.0,5369.59,79.49,130.0,3.9,89.8,490
4,PID000005,Home & Kitchen,Budget,40.62,27.0,1863.59,33.38,526.0,3.5,81.8,362


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 11 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Product_ID                         4000 non-null   object 
 1   Product_Category                   3874 non-null   object 
 2   Brand_Tier                         3944 non-null   object 
 3   Unit_Price                         3895 non-null   float64
 4   Discount_Percentage                3926 non-null   float64
 5   Advertising_Spend                  3876 non-null   float64
 6   Competitor_Price                   3936 non-null   float64
 7   Store_Coverage_Count               3928 non-null   float64
 8   Customer_Rating                    3920 non-null   float64
 9   Inventory_Availability_Percentage  3907 non-null   float64
 10  Monthly_Units_Sold                 4000 non-null   int64  
dtypes: float64(7), int64(1), object(3)
memory usage: 343.9+ 

## STEP 1: Check for Obvious Data-Entry Issues
Leading/trailing whitespace on all text columns, incorrect data types,
and stray symbols (e.g. "$", "%", ",") embedded in numeric-looking columns.

In [3]:
# Check for leading/trailing whitespace across ALL object/text columns
all_text_cols = df.select_dtypes(include=["object"]).columns
whitespace_issue_cols = []

for col in all_text_cols:
    has_whitespace = df[col].astype(str).apply(lambda x: x != x.strip()).sum()
    if has_whitespace > 0:
        whitespace_issue_cols.append(col)
        print(f"{col}: {has_whitespace} value(s) with leading/trailing whitespace")

In [4]:
# --- CORRECT ---
# Strip leading/trailing whitespace from all remaining text columns
for col in all_text_cols:
    df[col] = df[col].astype(str).str.strip()

In [5]:
# --- VERIFY ---
for col in all_text_cols:
    remaining_whitespace = df[col].apply(lambda x: x != x.strip()).sum()
    print(f"{col}: whitespace issues remaining =", remaining_whitespace)

Product_ID: whitespace issues remaining = 0
Product_Category: whitespace issues remaining = 0
Brand_Tier: whitespace issues remaining = 0


In [6]:
# --- IDENTIFY ---
# Check numeric-looking text columns for embedded symbols (currency signs,
# percent signs, thousands separators) that force the column to stay as text
symbol_issue_cols = []

for col in all_text_cols:
    sample_has_symbols = df[col].astype(str).str.contains(r"[\$,%]", regex=True, na=False).sum()
    if sample_has_symbols > 0:
        symbol_issue_cols.append(col)
        print(f"{col}: {sample_has_symbols} value(s) contain currency/percent/comma symbols")
        print(df[col][df[col].astype(str).str.contains(r"[\$,%]", regex=True, na=False)].head())

In [7]:
# --- CORRECT ---
# Remove stray symbols and convert these columns to numeric where possible
for col in symbol_issue_cols:
    cleaned = df[col].astype(str).str.replace(r"[\$,%]", "", regex=True).str.strip()
    converted = pd.to_numeric(cleaned, errors="coerce")
    # Only apply the conversion if it did not introduce excessive new NaNs
    if converted.notnull().sum() >= df[col].notnull().sum() * 0.9:
        df[col] = converted
        print(f"{col}: converted to numeric dtype")

## Step 2: Check for Missing Values

In [8]:
df = df.replace(
    ["nan", "NaN", "None", "null", "NULL", "NaT", ""],
    np.nan
)

In [9]:
# --- IDENTIFY ---
# Count of missing values per column
missing_counts = df.isnull().sum()
missing_percent = (df.isnull().sum() / len(df)) * 100

missing_summary = pd.DataFrame({
    "Missing_Count": missing_counts,
    "Missing_Percent": missing_percent.round(2)
})
missing_summary = missing_summary[missing_summary["Missing_Count"] > 0].sort_values(
    "Missing_Count", ascending=False
)
print("Columns with missing values:")
missing_summary

Columns with missing values:


,Missing_Count,Missing_Percent
Product_Category,126,3.15
Advertising_Spend,124,3.10
Unit_Price,105,2.62
Inventory_Availability_Percentage,93,2.33
Customer_Rating,80,2.00
Discount_Percentage,74,1.85
Store_Coverage_Count,72,1.80
Competitor_Price,64,1.60
Brand_Tier,56,1.40


In [10]:
# Display the actual rows that contain at least one missing value
rows_with_missing = df[df.isnull().any(axis=1)]
print("Number of rows with at least one missing value:", len(rows_with_missing))
rows_with_missing.head(10)

Number of rows with at least one missing value: 725


,Product_ID,Product_Category,Brand_Tier,Unit_Price,Discount_Percentage,Advertising_Spend,Competitor_Price,Store_Coverage_Count,Customer_Rating,Inventory_Availability_Percentage,Monthly_Units_Sold
29,PID000030,Sports & Fitness,Premium,107.79,NaN,8864.44,19.19,152.0,4.5,84.8,384
35,PID000036,NaN,Premium,24.06,5.7,NaN,150.05,121.0,4.2,76.2,217
55,PID000056,Sports & Fitness,NaN,83.56,13.7,2017.53,19.19,322.0,3.8,84.2,155
61,PID000062,Toys & Games,Premium,51.15,NaN,6158.11,47.39,196.0,4.3,95.1,335
62,PID000063,Apparel,Budget,26.04,24.9,NaN,95.41,592.0,3.8,87.1,543
67,PID000068,Toys & Games,Premium,NaN,NaN,5489.55,53.51,123.0,4.5,132.8,378
71,PID000072,NaN,Budget,NaN,23.9,1891.77,206.30,559.0,3.6,81.0,295
77,PID000078,Furniture,Premium,274.60,5.4,NaN,323.64,132.0,NaN,73.2,276
91,PID000092,NaN,Mid-range,32.19,18.4,6003.56,38.10,257.0,3.3,91.6,291
103,PID000104,Grocery,MID-RANGE,19.98,9.2,NaN,61.68,263.0,4.3,79.8,243


In [11]:
# --- CORRECT ---
# Numeric columns -> fill missing values with the column median
# (median is robust to outliers/skew compared to the mean)
numeric_cols = df.select_dtypes(include=[np.number]).columns

for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        median_value = df[col].median()
        df[col] = df[col].fillna(median_value)
        print(f"Filled {col} missing values with median = {median_value}")

Filled Unit_Price missing values with median = 43.21
Filled Discount_Percentage missing values with median = 14.0
Filled Advertising_Spend missing values with median = 3941.02
Filled Competitor_Price missing values with median = 76.785
Filled Store_Coverage_Count missing values with median = 266.0
Filled Customer_Rating missing values with median = 3.8
Filled Inventory_Availability_Percentage missing values with median = 85.2


In [12]:
# Categorical / object columns -> fill missing values with the column mode
# (if no mode exists, e.g. all missing, fall back to 'Unknown')
categorical_cols = df.select_dtypes(include=["object"]).columns

for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        if not df[col].mode().empty:
            mode_value = df[col].mode()[0]
        else:
            mode_value = "Unknown"
        df[col] = df[col].fillna(mode_value)
        print(f"Filled {col} missing values with mode = {mode_value}")

Filled Product_Category missing values with mode = Apparel
Filled Brand_Tier missing values with mode = Mid-range


In [13]:
# --- VERIFY ---
print("Remaining missing values per column:")
print(df.isnull().sum())
print("\nTotal remaining missing values:", df.isnull().sum().sum())

Remaining missing values per column:
Product_ID                           0
Product_Category                     0
Brand_Tier                           0
Unit_Price                           0
Discount_Percentage                  0
Advertising_Spend                    0
Competitor_Price                     0
Store_Coverage_Count                 0
Customer_Rating                      0
Inventory_Availability_Percentage    0
Monthly_Units_Sold                   0
dtype: int64

Total remaining missing values: 0


## STEP 3: Check for Duplicate Records

In [14]:
# --- IDENTIFY ---
num_duplicates = df.duplicated().sum()
print("Number of fully duplicated rows:", num_duplicates)

duplicate_rows = df[df.duplicated(keep=False)]
duplicate_rows.sort_values(by=list(df.columns)).head(10)

Number of fully duplicated rows: 6


,Product_ID,Product_Category,Brand_Tier,Unit_Price,Discount_Percentage,Advertising_Spend,Competitor_Price,Store_Coverage_Count,Customer_Rating,Inventory_Availability_Percentage,Monthly_Units_Sold
87,PID000088,Grocery,Budget,9.96,47.3,150.00,64.73,234.0,3.4,77.7,198
2631,PID000088,Grocery,Budget,9.96,47.3,150.00,64.73,234.0,3.4,77.7,198
1704,PID001705,Sports & Fitness,Budget,52.23,26.8,1539.03,98.51,334.0,3.5,92.8,233
3248,PID001705,Sports & Fitness,Budget,52.23,26.8,1539.03,98.51,334.0,3.5,92.8,233
484,PID003710,Toys & Games,Budget,17.27,22.2,3453.29,96.40,587.0,3.1,85.2,299
3709,PID003710,Toys & Games,Budget,17.27,22.2,3453.29,96.40,587.0,3.1,85.2,299
2378,PID003775,Toys & Games,Mid-range,32.14,22.7,8814.82,69.21,373.0,3.8,91.9,660
3774,PID003775,Toys & Games,Mid-range,32.14,22.7,8814.82,69.21,373.0,3.8,91.9,660
3297,PID003847,Electronic,Mid-range,223.39,4.9,4027.38,41.12,139.0,4.2,87.5,5
3846,PID003847,Electronic,Mid-range,223.39,4.9,4027.38,41.12,139.0,4.2,87.5,5


In [15]:
# --- CORRECT ---
# Remove duplicate rows, keeping the first occurrence
df = df.drop_duplicates(keep="first").reset_index(drop=True)

In [16]:
# --- VERIFY ---
print("Number of duplicate rows after removal:", df.duplicated().sum())
print("New shape of dataset:", df.shape)

Number of duplicate rows after removal: 0
New shape of dataset: (3994, 11)


## STEP 4: Check for Impossible or Invalid Values

In [17]:
# --- IDENTIFY: Price columns should not be negative ---
price_cols = [c for c in df.columns if "price" in c.lower()]
print("Detected price columns:", price_cols)

for col in price_cols:
    invalid_price = df[df[col] < 0]
    print(f"\n{col}: {len(invalid_price)} negative value(s) found")
    print(invalid_price[[col]].head())

Detected price columns: ['Unit_Price', 'Competitor_Price']

Unit_Price: 6 negative value(s) found
      Unit_Price
589       -18.00
645       -61.10
972       -19.55
2142      -76.42
2890      -10.80

Competitor_Price: 0 negative value(s) found
Empty DataFrame
Columns: [Competitor_Price]
Index: []


In [18]:
# --- CORRECT ---
# Negative prices are treated as a sign data-entry error -> take absolute value
for col in price_cols:
    df[col] = df[col].abs()

In [19]:
# --- VERIFY ---
for col in price_cols:
    print(f"{col}: negative values remaining =", (df[col] < 0).sum())

Unit_Price: negative values remaining = 0
Competitor_Price: negative values remaining = 0


In [20]:
# --- IDENTIFY: Discount percentage should be between 0 and 100 ---
discount_cols = [c for c in df.columns if "discount" in c.lower()]
print("Detected discount columns:", discount_cols)

for col in discount_cols:
    invalid_discount = df[(df[col] < 0) | (df[col] > 100)]
    print(f"\n{col}: {len(invalid_discount)} out-of-range value(s) found")
    print(invalid_discount[[col]].head())

Detected discount columns: ['Discount_Percentage']

Discount_Percentage: 5 out-of-range value(s) found
      Discount_Percentage
328                 128.1
1416                156.0
2275                130.7
2786                144.9
2889                138.9


In [21]:
# --- CORRECT ---
# Clip discount percentage values to the valid 0-65 range
for col in discount_cols:
    df[col] = df[col].clip(lower=0, upper=65)

In [22]:
# --- VERIFY ---
for col in discount_cols:
    out_of_range = ((df[col] < 0) | (df[col] > 100)).sum()
    print(f"{col}: out-of-range values remaining =", out_of_range)

Discount_Percentage: out-of-range values remaining = 0


In [23]:
# --- IDENTIFY: Customer rating should be between 1 and 5 ---
rating_cols = [c for c in df.columns if "rating" in c.lower()]
print("Detected rating columns:", rating_cols)

for col in rating_cols:
    invalid_rating = df[(df[col] < 1) | (df[col] > 5)]
    print(f"\n{col}: {len(invalid_rating)} out-of-range value(s) found")
    print(invalid_rating[[col]].head())

Detected rating columns: ['Customer_Rating']

Customer_Rating: 6 out-of-range value(s) found
      Customer_Rating
1209              6.5
1912              6.6
2363              6.5
2791              0.2
3392              6.2


In [24]:
# --- CORRECT ---
# Clip rating values to the valid 1-5 range
for col in rating_cols:
    df[col] = df[col].clip(lower=1, upper=5)

In [25]:
# --- VERIFY ---
for col in rating_cols:
    out_of_range = ((df[col] < 1) | (df[col] > 5)).sum()
    print(f"{col}: out-of-range values remaining =", out_of_range)

Customer_Rating: out-of-range values remaining = 0


In [26]:
# --- IDENTIFY: Inventory availability should be between 0 and 100 % ---
availability_cols = [
    c for c in df.columns if "availability" in c.lower() or "inventory" in c.lower()
]
print("Detected inventory/availability columns:", availability_cols)

for col in availability_cols:
    invalid_availability = df[(df[col] < 0) | (df[col] > 100)]
    print(f"\n{col}: {len(invalid_availability)} out-of-range value(s) found")
    print(invalid_availability[[col]].head())

Detected inventory/availability columns: ['Inventory_Availability_Percentage']

Inventory_Availability_Percentage: 6 out-of-range value(s) found
      Inventory_Availability_Percentage
67                                132.8
877                               109.7
2232                              123.5
2316                              119.6
2645                              125.8


In [27]:
# --- CORRECT ---
# Clip inventory availability values to the valid 0-100 range
for col in availability_cols:
    df[col] = df[col].clip(lower=0, upper=100)

In [28]:
# --- VERIFY ---
for col in availability_cols:
    out_of_range = ((df[col] < 0) | (df[col] > 100)).sum()
    print(f"{col}: out-of-range values remaining =", out_of_range)

Inventory_Availability_Percentage: out-of-range values remaining = 0


## STEP 5: Check for Inconsistent Categorical Values
Focus on categorical columns such as Product_Category and Brand_Tier.

In [29]:
# --- IDENTIFY ---
# List the categorical columns actually present in this dataset
category_like_cols = [
    c for c in df.columns if "category" in c.lower() or "tier" in c.lower() or "brand" in c.lower()
]
print("Detected categorical columns to standardize:", category_like_cols)

for col in category_like_cols:
    print(f"\nUnique values in '{col}' BEFORE cleaning ({df[col].nunique()} unique):")
    print(sorted(df[col].dropna().unique()))

Detected categorical columns to standardize: ['Product_Category', 'Brand_Tier']

Unique values in 'Product_Category' BEFORE cleaning (30 unique):
['APPAREL', 'Apparel', 'Apparrel', 'BEAUTY & PERSONAL CARE', 'Beauty & Personal Care', 'Beauty and Personal Care', 'ELECTRONICS', 'Electronic', 'Electronics', 'FURNITURE', 'Furniture', 'Furnitures', 'GROCERY', 'Groceries', 'Grocery', 'HOME & KITCHEN', 'Home & Kitchen', 'Home and Kitchen', 'Sports & Fitness', 'Sports and Fitness', 'TOYS & GAMES', 'Toys & Games', 'Toys and Games', 'apparel', 'beauty & personal care', 'electronics', 'grocery', 'home & kitchen', 'sports & fitness', 'toys & games']

Unique values in 'Brand_Tier' BEFORE cleaning (12 unique):
['BUDGET', 'Budget', 'Budgett', 'MID-RANGE', 'Mid-range', 'Midrange', 'PREMIUM', 'Premeium', 'Premium', 'budget', 'mid-range', 'premium']


In [30]:
# --- CORRECT ---
# Standardize: strip leading/trailing whitespace, collapse internal double spaces,
# and apply consistent Title Case capitalization so that values like
# "electronics", "Electronics ", " ELECTRONICS" all map to "Electronics"
for col in category_like_cols:
    df[col] = df[col].astype(str).str.strip()
    df[col] = df[col].str.replace(r"\s+", " ", regex=True)
    df[col] = df[col].str.title()

# Standardizing the "Product_Category" column
product_category_fixes = {
    "Apparrel": "Apparel",
    "Beauty And Personal Care": "Beauty & Personal Care",
    "Electronic": "Electronics",
    "Electronicss": "Electronics",
    "Furnitures": "Furniture",
    "Groceries": "Grocery",
    "Home And Kitchen": "Home & Kitchen",
    "Sports And Fitness": "Sports & Fitness",
    "Toys And Games": "Toys & Games",
}

df["Product_Category"] = df["Product_Category"].replace(product_category_fixes)

# Standardizing the "Brand_Tier" column
brand_tier_fixes = {
    "Budgett": "Budget",
    "Midrange": "Mid-Range",
    "Premeium": "Premium",
}

df["Brand_Tier"] = df["Brand_Tier"].replace(brand_tier_fixes)

In [31]:
# --- VERIFY ---
for col in category_like_cols:
    print(f"\nUnique values in '{col}' AFTER cleaning ({df[col].nunique()} unique):")
    print(sorted(df[col].dropna().unique()))


Unique values in 'Product_Category' AFTER cleaning (8 unique):
['Apparel', 'Beauty & Personal Care', 'Electronics', 'Furniture', 'Grocery', 'Home & Kitchen', 'Sports & Fitness', 'Toys & Games']

Unique values in 'Brand_Tier' AFTER cleaning (3 unique):
['Budget', 'Mid-Range', 'Premium']


## Final Check

In [32]:
# --- VERIFY ---
print("Data types after all corrections steps:")
print(df.dtypes)
print("\nRemaining missing values after all cleaning steps:")
print(df.isnull().sum())

Data types after all corrections steps:
Product_ID                            object
Product_Category                      object
Brand_Tier                            object
Unit_Price                           float64
Discount_Percentage                  float64
Advertising_Spend                    float64
Competitor_Price                     float64
Store_Coverage_Count                 float64
Customer_Rating                      float64
Inventory_Availability_Percentage    float64
Monthly_Units_Sold                     int64
dtype: object

Remaining missing values after all cleaning steps:
Product_ID                           0
Product_Category                     0
Brand_Tier                           0
Unit_Price                           0
Discount_Percentage                  0
Advertising_Spend                    0
Competitor_Price                     0
Store_Coverage_Count                 0
Customer_Rating                      0
Inventory_Availability_Percentage    0
Monthly_U

In [33]:
# --- FINAL CHECK ---
print("Final dataset shape:", df.shape)
df.head()

Final dataset shape: (3994, 11)


,Product_ID,Product_Category,Brand_Tier,Unit_Price,Discount_Percentage,Advertising_Spend,Competitor_Price,Store_Coverage_Count,Customer_Rating,Inventory_Availability_Percentage,Monthly_Units_Sold
0,PID000001,Beauty & Personal Care,Premium,58.46,1.7,9748.70,47.46,88.0,4.3,87.9,389
1,PID000002,Grocery,Mid-Range,15.98,17.9,5279.12,48.54,215.0,3.8,99.5,450
2,PID000003,Beauty & Personal Care,Budget,16.35,17.7,5165.84,59.57,481.0,3.5,96.7,594
3,PID000004,Grocery,Mid-Range,11.31,34.0,5369.59,79.49,130.0,3.9,89.8,490
4,PID000005,Home & Kitchen,Budget,40.62,27.0,1863.59,33.38,526.0,3.5,81.8,362


# Model Building

In [34]:
target = 'Monthly_Units_Sold'
id_col = 'Product_ID'   # not a real feature, just an identifier

numerical_cols = df.select_dtypes(include='number').columns.tolist()
numerical_cols.remove(target)

categorical_cols = df.select_dtypes(include='object').columns.tolist()
categorical_cols.remove(id_col)

print("Numerical:", numerical_cols)
print("Categorical:", categorical_cols)

Numerical: ['Unit_Price', 'Discount_Percentage', 'Advertising_Spend', 'Competitor_Price', 'Store_Coverage_Count', 'Customer_Rating', 'Inventory_Availability_Percentage']
Categorical: ['Product_Category', 'Brand_Tier']


In [35]:
# Build X and y
X = pd.concat([df[numerical_cols], pd.get_dummies(df[categorical_cols], drop_first=True)], axis=1)
y = df[target]

In [36]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(    
    X, y,    
    test_size=0.20,    
    random_state=42)

print("Train:", X_train.shape, "Test:", X_test.shape)

Train: (3195, 16) Test: (799, 16)


In [37]:
from sklearn.linear_model import LinearRegression

regr = LinearRegression()

regr.fit(X_train, y_train)

y_pred = regr.predict(X_test)

In [38]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(y_test, y_pred)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))

r2 = r2_score(y_test, y_pred)

print("MAE :", mae)
print("RMSE:", rmse)
print("R²  :", r2)

MAE : 45.75139410340704
RMSE: 59.93166747182324
R²  : 0.8899272714784167


In [39]:
# Mean of residuals
residuals = y_test.values - y_pred
print("Mean of Residuals:", np.mean(residuals))

Mean of Residuals: -5.61913271985134


In [40]:
import pickle

with open("linear_regression_model.pkl", "wb") as file:    
    pickle.dump(regr, file)

print("Model saved to linear_regression_model.pkl")

Model saved to linear_regression_model.pkl


In [41]:
!pip install flask pandas numpy scikit-learn

In [ ]:
!python app.py